# Thí nghiệm B — Qwen3-14B + LoRA (model chính)

Giống hệt thí nghiệm A (cùng dữ liệu, siêu tham số, batch hiệu dụng 16, seed), chỉ đổi model sang **Qwen3-14B**.
Batch mỗi bước nhỏ hơn (4 × 4 thay vì 8 × 2) để vừa bộ nhớ; batch hiệu dụng không đổi.

Cần: GPU **A100 80GB**, Colab Secret `HF_TOKEN`, đã chạy notebook **00**. Thời gian ước tính: tải model ~8 phút, train 3 epoch ~30–45 phút, suy luận ~8 phút.

In [ ]:
#@title 1. Kiểm tra GPU
import subprocess
info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                       '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout.strip()
print(info)
name, mem = [x.strip() for x in info.split(',')]
assert 'A100' in name and int(mem) >= 79000, (
    f'Cấu hình batch trong notebook tính cho A100 80GB, runtime hiện tại: {name} {mem} MiB')

In [ ]:
#@title 2. Cấu hình thí nghiệm
EXP = 'expB_qwen3_14b'  #@param {type:'string'}
MODEL = 'Qwen/Qwen3-14B'  #@param {type:'string'}
EPOCHS = 3  #@param {type:'number'}
LR = 1e-4  #@param {type:'number'}
RANK = 32  #@param {type:'integer'}
ALPHA = 64  #@param {type:'integer'}
BATCH = 4  #@param {type:'integer'}
GRAD_ACCUM = 4  #@param {type:'integer'}
TASKS = 'gen,solve'  #@param ['gen,solve', 'gen']
SHUFFLE_AUG = 0  #@param {type:'integer'}
SEED = 42  #@param {type:'integer'}
GEN_SAMPLES = 2  #@param {type:'integer'}
SYSTEMS = 'base,base_fs3,lora'  #@param {type:'string'}
REPO = 'https://github.com/trantrien1/AQG.git'  #@param {type:'string'}
BRANCH = 'lora-finetune'  #@param {type:'string'}
DRIVE_ROOT = '/content/drive/MyDrive/AQG_ft'  #@param {type:'string'}

DATA_DIR = f'{DRIVE_ROOT}/data'
RUN = f'{EXP}_seed{SEED}'
WORK = f'/content/{RUN}'            # checkpoint ghi đĩa cục bộ cho nhanh
SAVE = f'{DRIVE_ROOT}/{RUN}'         # adapter, tóm tắt, dự đoán chép lên Drive
print('Batch hiệu dụng:', BATCH * GRAD_ACCUM)

In [ ]:
#@title 3. Drive, token Hugging Face, hàm chạy lệnh
import os, sys, json, time, shutil, subprocess
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_ROOT, exist_ok=True)

try:  # Colab: 🔑 Secrets -> thêm HF_TOKEN (cần quyền đọc dataset riêng tư)
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception as exc:
    print('Chưa đọc được HF_TOKEN từ Colab Secrets:', exc)

REPO_DIR = '/content/AQG'
NB_DIR = f'{REPO_DIR}/API/notebooks'
ENV = dict(os.environ, PYTHONPATH=NB_DIR, TOKENIZERS_PARALLELISM='false',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

def sh(cmd, log=None):
    """Chạy lệnh, in đầu ra ngay khi có; lỗi thì dừng notebook."""
    print('$', cmd, flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    p = subprocess.Popen(cmd, shell=True, cwd=NB_DIR if os.path.isdir(NB_DIR) else None,
                         env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, encoding='utf-8', errors='replace')
    for chunk in iter(lambda: p.stdout.read(512), ''):
        sys.stdout.write(chunk)
        if fh:
            fh.write(chunk)
    if fh:
        fh.close()
    if p.wait() != 0:
        raise RuntimeError(f'Lệnh lỗi (mã {p.returncode}): {cmd}')

In [ ]:
#@title 4. Clone repo + cài thư viện (~5 phút)
subprocess.run(['rm', '-rf', REPO_DIR], check=True)
sh(f'git clone -q --depth 1 -b {BRANCH} {REPO} {REPO_DIR} && git -C {REPO_DIR} log --oneline -1')
# vLLM cài trước vì nó ghim phiên bản torch; peft không ghim torch.
sh('pip -q install -U vllm && pip -q install -U "peft>=0.15" pytest')
sh('python -c "import torch, transformers, peft, vllm; '
   'print(\'torch\', torch.__version__, \'| transformers\', transformers.__version__, '
   '\'| peft\', peft.__version__, \'| vllm\', vllm.__version__)"')
sh('python -m pytest -q ../tests/test_mcqft.py')

In [ ]:
#@title 5. Ghim phiên bản trọng số, tải trước model, kiểm tra dữ liệu
from huggingface_hub import HfApi, snapshot_download
REVISION = HfApi().model_info(MODEL).sha
print(MODEL, '@', REVISION)
snapshot_download(MODEL, revision=REVISION, allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'])
assert os.path.exists(f'{DATA_DIR}/split.json'), 'Chưa có dữ liệu: chạy notebook 00 trước'
print(open(f'{DATA_DIR}/stats.md', encoding='utf-8').read())
os.makedirs(SAVE, exist_ok=True)
json.dump({'exp': EXP, 'model': MODEL, 'revision': REVISION, 'epochs': EPOCHS, 'lr': LR,
           'rank': RANK, 'alpha': ALPHA, 'batch': BATCH, 'grad_accum': GRAD_ACCUM,
           'tasks': TASKS, 'shuffle_aug': SHUFFLE_AUG, 'seed': SEED,
           'branch': BRANCH, 'started': time.strftime('%Y-%m-%d %H:%M:%S')},
          open(f'{SAVE}/config.json', 'w'), indent=1)

## Huấn luyện LoRA

bf16, không lượng tử hoá (A100 80GB đủ chỗ). LoRA gắn vào mọi lớp tuyến tính của attention và MLP.
Loss chỉ tính trên câu trả lời. Sau mỗi epoch đánh giá loss trên tập val; adapter cuối là checkpoint có val loss thấp nhất.

In [ ]:
#@title 6. Train
sh(f'python -m mcqft.train --model {MODEL} --revision {REVISION} --data "{DATA_DIR}" '
   f'--out {WORK} --tasks {TASKS} --shuffle-aug {SHUFFLE_AUG} --epochs {EPOCHS} --lr {LR} '
   f'--rank {RANK} --alpha {ALPHA} --batch {BATCH} --grad-accum {GRAD_ACCUM} --seed {SEED}',
   log=f'{SAVE}/train.log')
shutil.copytree(f'{WORK}/adapter', f'{SAVE}/adapter', dirs_exist_ok=True)
shutil.copy(f'{WORK}/train_summary.json', SAVE)

In [ ]:
#@title 7. Đường loss
import matplotlib.pyplot as plt
summary = json.load(open(f'{SAVE}/train_summary.json'))
hist = summary['log_history']
tr = [(h['step'], h['loss']) for h in hist if 'loss' in h]
ev = [(h['step'], h['eval_loss']) for h in hist if 'eval_loss' in h]
plt.figure(figsize=(7, 3.5))
plt.plot(*zip(*tr), label='train')
plt.plot(*zip(*ev), 'o-', label='val')
plt.xlabel('bước'); plt.ylabel('loss'); plt.title(f'{EXP}: {MODEL}'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{SAVE}/loss.png', dpi=150, bbox_inches='tight'); plt.show()
print({k: summary[k] for k in ('train_seconds', 'train_tokens_per_second', 'peak_gpu_mem_gb',
                               'best_eval_loss', 'best_checkpoint')})

## Sinh đầu ra trên tập test

Ba hệ trên cùng các câu test:

| Hệ | Ý nghĩa |
|---|---|
| `base` | model gốc, không ví dụ mẫu |
| `base_fs3` | model gốc + 3 ví dụ mẫu lấy từ train (baseline mạnh hơn cho tác vụ sinh) |
| `lora` | model gốc + adapter vừa train |

Tác vụ **gen**: mỗi câu test cho ra `GEN_SAMPLES` câu mới cùng chủ đề/dạng/mức độ. Tác vụ **solve**: giải đúng câu test (greedy).

In [ ]:
#@title 8. Suy luận bằng vLLM (vài phút)
sh(f'python -m mcqft.infer --model {MODEL} --revision {REVISION} --data "{DATA_DIR}" '
   f'--adapter {WORK}/adapter --out {WORK}/preds --systems {SYSTEMS} --tasks {TASKS} '
   f'--gen-samples {GEN_SAMPLES} --seed {SEED}', log=f'{SAVE}/infer.log')
shutil.copytree(f'{WORK}/preds', f'{SAVE}/preds', dirs_exist_ok=True)

In [ ]:
#@title Cài KaTeX (kiểm tra công thức trong câu sinh ra)
sh('mkdir -p /content/katex && cd /content/katex && npm install --silent katex@0.16 && ls node_modules | head -3')
KATEX_MODULES = '/content/katex/node_modules'

In [ ]:
#@title 9. Chỉ số nhanh (chưa có giám khảo — chạy notebook 03 để có số cuối)
from IPython.display import Markdown, display
sh(f'python -m mcqft.report --data "{DATA_DIR}" --exp {EXP}="{SAVE}" '
   f'--out "{SAVE}/report_quick" --katex-modules {KATEX_MODULES}')
display(Markdown(open(f'{SAVE}/report_quick/report.md', encoding='utf-8').read()))

In [ ]:
#@title 10. Xem vài câu LoRA sinh ra
sys.path.insert(0, NB_DIR)
from mcqft.data import read_jsonl
rows = read_jsonl(f'{SAVE}/preds/lora.gen.jsonl')
for r in rows[:3]:
    print('=' * 20, r['id'], r['finish_reason'])
    print(r['text'][:1500])

In [ ]:
#@title 11. (Tuỳ chọn) Đẩy adapter lên Hugging Face ở chế độ riêng tư
PUSH = False  #@param {type:'boolean'}
HF_REPO = ''  #@param {type:'string'}
HF_REPO = HF_REPO or f'trantrien1/aqg-{EXP}-lora'
if PUSH:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=HF_REPO, folder_path=f'{SAVE}/adapter')
    api.upload_file(repo_id=HF_REPO, path_or_fileobj=f'{SAVE}/train_summary.json',
                    path_in_repo='train_summary.json')
    print('Đã đẩy:', HF_REPO)